In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 4-Layer Hierarchical Multi-Class Logistic Regressor (`models/multi_lr.ipynb`)

This notebook implements the complete **4-Layer Hierarchical Cascade Logistic Regression System** using straight `if-else` branching output logic specified in `TODO.md` section `## Structure`:

### Sequential Decision Logic
1. **Layer 1 (`lr_feng_esi1_extreme_model.rds`)**: Predicts **ESI 1 vs. Not ESI 1**.
   - `if (Layer 1 output == "1")` -> Predict **ESI 1**.
2. **Layer 2 (`lr_feng_esi5_extreme_model.rds`)**: Predicts **ESI 5 vs. Not ESI 5**.
   - `else if (Layer 2 output == "5")` -> Predict **ESI 5**.
3. **Layer 3 (`lr_feng_esi2_model.rds`)**: Predicts **ESI 2 vs. Not ESI 2**.
   - `else if (Layer 3 output == "2")` -> Predict **ESI 2**.
4. **Layer 4 (`lr_feng_esi34_model.rds`)**: Predicts **ESI 3 vs. ESI 4**.
   - `else` -> Predict Layer 4 output (**ESI 3** or **ESI 4**).

### Evaluation Protocol
- Evaluated on the **Complete Test Distribution Set** (all 5 ESI levels in their natural population frequencies).
- Reports **5x5 Confusion Matrix**, **Accuracy**, **Precision**, **Recall (Sensitivity)**, **PR-AUC**, and **ROC-AUC**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Complete .RData Dataset & Construct 13 FE Features
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading complete dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender + cc_breathingdifficulty
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_feng <- na.omit(df_feng)

cat(sprintf("Complete Feature Engineered Dataset: %d rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("5-Class ESI Distribution in Full Population:\n")
print(table(df_feng$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning (Extract Complete 15% Test Distribution Set)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$raw_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$raw_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows (Complete Distribution Set)\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))
cat("\nTest Set Class Distribution (Complete Set):\n")
print(table(test_df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Load Component Layer Models (or Train Fallback if Not Saved Yet)
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

path_m1 <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
path_m2 <- file.path(deploy_dir, "lr_feng_esi5_extreme_model.rds")
path_m3 <- file.path(deploy_dir, "lr_feng_esi2_model.rds")
path_m4 <- file.path(deploy_dir, "lr_feng_esi34_model.rds")

# --- Train Layer 1 if missing ---
if (!file.exists(path_m1)) {
  cat("Training Layer 1 Model (ESI 1 vs Not 1)...\n")
  df_m1 <- train_df
  df_m1$target <- factor(ifelse(df_m1$raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
  pp_1 <- preProcess(df_m1[, "age", drop = FALSE], method = c("center", "scale"))
  df_m1 <- predict(pp_1, df_m1)
  m1 <- multinom(target ~ age + gender + cc_breathingdifficulty + is_dyspnea_total + is_dyspnea_moderate + is_bradypnea + is_tachypnea + is_hypotension + is_hypertension + is_bradycardia_total + is_bradycardia_moderate + is_tachycardia_total + is_tachycardia_moderate, data = df_m1, trace = FALSE)
  mod1 <- list(model = m1, preproc = pp_1)
  saveRDS(mod1, path_m1)
} else {
  mod1 <- readRDS(path_m1)
}

# --- Train Layer 2 if missing ---
if (!file.exists(path_m2)) {
  cat("Training Layer 2 Model (ESI 5 vs Not 5)...\n")
  df_m2 <- train_df[train_df$raw_esi != "1", ]
  df_m2$target <- factor(ifelse(df_m2$raw_esi == "5", "5", "not_5"), levels = c("5", "not_5"))
  pp_2 <- preProcess(df_m2[, "age", drop = FALSE], method = c("center", "scale"))
  df_m2 <- predict(pp_2, df_m2)
  m2 <- multinom(target ~ age + gender + cc_breathingdifficulty + is_dyspnea_total + is_dyspnea_moderate + is_bradypnea + is_tachypnea + is_hypotension + is_hypertension + is_bradycardia_total + is_bradycardia_moderate + is_tachycardia_total + is_tachycardia_moderate, data = df_m2, trace = FALSE)
  mod2 <- list(model = m2, preproc = pp_2)
  saveRDS(mod2, path_m2)
} else {
  mod2 <- readRDS(path_m2)
}

# --- Train Layer 3 if missing ---
if (!file.exists(path_m3)) {
  cat("Training Layer 3 Model (ESI 2 vs Not 2)...\n")
  df_m3 <- train_df[!(train_df$raw_esi %in% c("1", "5")), ]
  df_m3$target <- factor(ifelse(df_m3$raw_esi == "2", "2", "not_2"), levels = c("2", "not_2"))
  pp_3 <- preProcess(df_m3[, "age", drop = FALSE], method = c("center", "scale"))
  df_m3 <- predict(pp_3, df_m3)
  m3 <- multinom(target ~ age + gender + cc_breathingdifficulty + is_dyspnea_total + is_dyspnea_moderate + is_bradypnea + is_tachypnea + is_hypotension + is_hypertension + is_bradycardia_total + is_bradycardia_moderate + is_tachycardia_total + is_tachycardia_moderate, data = df_m3, trace = FALSE)
  mod3 <- list(model = m3, preproc = pp_3)
  saveRDS(mod3, path_m3)
} else {
  mod3 <- readRDS(path_m3)
}

# --- Train Layer 4 if missing ---
if (!file.exists(path_m4)) {
  cat("Training Layer 4 Model (ESI 3 vs ESI 4)...\n")
  df_m4 <- train_df[train_df$raw_esi %in% c("3", "4"), ]
  df_m4$target <- factor(df_m4$raw_esi, levels = c("3", "4"))
  pp_4 <- preProcess(df_m4[, "age", drop = FALSE], method = c("center", "scale"))
  df_m4 <- predict(pp_4, df_m4)
  m4 <- multinom(target ~ age + gender + cc_breathingdifficulty + is_dyspnea_total + is_dyspnea_moderate + is_bradypnea + is_tachypnea + is_hypotension + is_hypertension + is_bradycardia_total + is_bradycardia_moderate + is_tachycardia_total + is_tachycardia_moderate, data = df_m4, trace = FALSE)
  mod4 <- list(model = m4, preproc = pp_4)
  saveRDS(mod4, path_m4)
} else {
  mod4 <- readRDS(path_m4)
}

cat("All 4 Layer Models Successfully Loaded and Ready for Cascade Evaluation!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Straight If-Else Sequential Branching Decision Engine
# ---------------------------------------------------------
predict_straight_ifelse_lr <- function(data, mod1, mod2, mod3, mod4) {
  N <- nrow(data)
  
  # Standardize continuous features per model preprocessing transform
  d1 <- predict(mod1$preproc, data)
  d2 <- predict(mod2$preproc, data)
  d3 <- predict(mod3$preproc, data)
  d4 <- predict(mod4$preproc, data)
  
  # Get straight class predictions from each model
  pred1 <- as.character(predict(mod1$model, newdata = d1, type = "class"))
  pred2 <- as.character(predict(mod2$model, newdata = d2, type = "class"))
  pred3 <- as.character(predict(mod3$model, newdata = d3, type = "class"))
  pred4 <- as.character(predict(mod4$model, newdata = d4, type = "class"))
  
  # Get probabilities for metric evaluation (ROC-AUC / PR-AUC)
  prob1 <- predict(mod1$model, newdata = d1, type = "probs")
  prob2 <- predict(mod2$model, newdata = d2, type = "probs")
  prob3 <- predict(mod3$model, newdata = d3, type = "probs")
  prob4 <- predict(mod4$model, newdata = d4, type = "probs")
  
  p1 <- if (is.matrix(prob1)) prob1[, "1"] else prob1
  p5 <- if (is.matrix(prob2)) prob2[, "5"] else prob2
  p2 <- if (is.matrix(prob3)) prob3[, "2"] else prob3
  
  if (is.matrix(prob4)) {
    p3 <- prob4[, "3"]
    p4 <- prob4[, "4"]
  } else {
    p3 <- prob4
    p4 <- 1 - prob4
  }
  
  prob_matrix <- cbind("1" = p1, "2" = p2, "3" = p3, "4" = p4, "5" = p5)
  
  # Straight If-Else Branching Decision Rule
  pred_classes <- character(N)
  for (i in 1:N) {
    if (pred1[i] == "1") {
      pred_classes[i] <- "1"
    } else if (pred2[i] == "5") {
      pred_classes[i] <- "5"
    } else if (pred3[i] == "2") {
      pred_classes[i] <- "2"
    } else {
      pred_classes[i] <- pred4[i]  # "3" or "4"
    }
  }
  
  pred_factor <- factor(pred_classes, levels = c("1", "2", "3", "4", "5"))
  return(list(pred_factor = pred_factor, prob_matrix = prob_matrix))
}

cat("Straight if-else sequential branching decision engine initialized!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Complete Benchmark Evaluation on Test Set (Confusion Matrix, Acc, Prec, Rec, PR-AUC, ROC-AUC)
# ---------------------------------------------------------
# Function to compute PR-AUC (Precision-Recall Area Under Curve)
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
# Compute Predictions on Test Set using Straight If-Else Branching
res_test <- predict_straight_ifelse_lr(test_df, mod1, mod2, mod3, mod4)
pred_test <- res_test$pred_factor
prob_test <- res_test$prob_matrix
actual_test <- factor(test_df$raw_esi, levels = c("1", "2", "3", "4", "5"))

# 5x5 Confusion Matrix
cm <- confusionMatrix(pred_test, actual_test)
acc <- as.numeric(cm$overall["Accuracy"])

# Extract Per-Class Precision and Recall
prec_by_class <- cm$byClass[, "Pos Pred Value"]
rec_by_class  <- cm$byClass[, "Sensitivity"]
macro_prec    <- mean(prec_by_class, na.rm = TRUE)
macro_rec     <- mean(rec_by_class,  na.rm = TRUE)

# Compute Per-Class & Macro PR-AUC
pr_auc_by_class <- numeric(5)
names(pr_auc_by_class) <- c("1", "2", "3", "4", "5")
for (cls in c("1", "2", "3", "4", "5")) {
  act_bin <- ifelse(actual_test == cls, 1, 0)
  pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_test[, cls])
}
macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)

# Compute Multi-Class ROC-AUC using pROC
roc_obj <- pROC::multiclass.roc(actual_test, prob_test)
macro_roc_auc <- as.numeric(roc_obj$auc)

# Print Comprehensive Benchmark Summary
cat(sprintf("============================================================\n"))
cat(sprintf("   4-LAYER HIERARCHICAL LOGISTIC REGRESSOR - COMPLETE TEST SET BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
cat(sprintf("============================================================\n\n"))

cat("Per-Class Performance Summary Table:\n")
per_class_metrics <- data.frame(
  Class        = c("1", "2", "3", "4", "5"),
  Actual_Count = as.numeric(table(actual_test)),
  Pred_Count   = as.numeric(table(pred_test)),
  Precision    = round(prec_by_class, 4),
  Recall       = round(rec_by_class, 4),
  PR_AUC       = round(pr_auc_by_class, 4)
)
print(per_class_metrics)

cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm$table)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7A: Diagnostic Plot 1 - Per-Class Metrics Bar Chart
# ---------------------------------------------------------
metrics_long <- per_class_metrics %>%
  pivot_longer(cols = c("Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")

p_bar <- ggplot(metrics_long, aes(x = Class, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Precision" = "#2b5c8f", "Recall" = "#e07a5f", "PR_AUC" = "#81b29a")) +
  labs(title = "4-Layer Multi LR: Per-Class Performance (Test Set)",
       subtitle = "Comparing Precision, Recall, and PR-AUC across all 5 ESI levels",
       x = "ESI Level", y = "Metric Value") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "top")

if (!dir.exists("../plots")) dir.create("../plots", recursive = TRUE)
ggsave("../plots/multi_lr_metrics_barchart.png", plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Per-Class Metrics Bar Chart saved to: plots/multi_lr_metrics_barchart.png\n")

p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7B: Diagnostic Plot 2 - 5-Class Overlaid Precision-Recall (PR) Curves
# ---------------------------------------------------------
pr_df_list <- list()
for (cls in c("1", "2", "3", "4", "5")) {
  prob_pos <- prob_test[, cls]
  ord <- order(prob_pos, decreasing = TRUE)
  act_sorted <- (actual_test[ord] == cls)
  tp <- cumsum(act_sorted)
  fp <- cumsum(!act_sorted)
  n_pos <- sum(act_sorted)
  rec  <- c(0, tp / n_pos)
  prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
  pr_df_list[[cls]] <- data.frame(
    Recall = rec,
    Precision = prec,
    Class = sprintf("ESI %s (PR-AUC = %.3f)", cls, pr_auc_by_class[cls])
  )
}

df_pr_all <- do.call(rbind, pr_df_list)

p_pr <- ggplot(df_pr_all, aes(x = Recall, y = Precision, color = Class)) +
  geom_line(size = 1.2) +
  theme_minimal() +
  scale_color_manual(values = c("#d90429", "#f77f00", "#2a9d8f", "#457b9d", "#1d3557")) +
  labs(title = "4-Layer Multi LR: Precision-Recall Curves (Complete Test Set)",
       subtitle = "Overlaid PR curves for all 5 ESI triage levels",
       x = "Recall (Sensitivity)", y = "Precision (Positive Predictive Value)") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "bottom")

ggsave("../plots/multi_lr_pr_curves.png", plot = p_pr, width = 8, height = 5.5, dpi = 300)
cat("PR Curves plot saved to: plots/multi_lr_pr_curves.png\n")

p_pr

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7C: Diagnostic Plot 3 - 5-Class Overlaid ROC Curves
# ---------------------------------------------------------
roc_df_list <- list()
for (cls in c("1", "2", "3", "4", "5")) {
  r_obj <- pROC::roc(actual_test == cls, prob_test[, cls], quiet = TRUE)
  roc_df_list[[cls]] <- data.frame(
    FPR = 1 - r_obj$specificities,
    TPR = r_obj$sensitivities,
    Class = sprintf("ESI %s (AUC = %.3f)", cls, r_obj$auc)
  )
}

df_roc_all <- do.call(rbind, roc_df_list)

p_roc <- ggplot(df_roc_all, aes(x = FPR, y = TPR, color = Class)) +
  geom_line(size = 1.2) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "gray50") +
  theme_minimal() +
  scale_color_manual(values = c("#d90429", "#f77f00", "#2a9d8f", "#457b9d", "#1d3557")) +
  labs(title = "4-Layer Multi LR: ROC Curves Comparison (Complete Test Set)",
       subtitle = "Overlaid 1-vs-Rest ROC curves for all 5 ESI triage levels",
       x = "False Positive Rate (1 - Specificity)", y = "True Positive Rate (Sensitivity)") +
  theme(plot.title = element_text(face = "bold", size = 14), legend.position = "bottom")

ggsave("../plots/multi_lr_roc_curves.png", plot = p_roc, width = 8, height = 5.5, dpi = 300)
cat("ROC Curves plot saved to: plots/multi_lr_roc_curves.png\n")

p_roc